# ROGII run v7 full ensemble

Full run_v7 inference: PF128 + v10 TabICL raw offsets + ravaghi c50-c54 raw offsets.


In [ ]:
# Write helper modules into the Kaggle working directory before imports.
from pathlib import Path

Path("koolbox/trainer").mkdir(parents=True, exist_ok=True)
Path("koolbox/__init__.py").write_text("from .trainer.trainer import Trainer\n")
Path("koolbox/trainer/__init__.py").write_text("from .trainer import Trainer\n")
Path("koolbox/trainer/trainer.py").write_text("import os\nimport joblib\nimport numpy as np\nfrom sklearn.base import clone\n\nclass Trainer:\n    def __init__(self, estimator=None, task=\"regression\", metric=None, cv=None,\n                 cv_args=None, use_early_stopping=False, verbose=True, save=False,\n                 save_path=None, metric_precision=5, metric_threshold=None,\n                 metric_args=None, **kwargs):\n        self.estimator = estimator\n        self.estimator_name = estimator.__class__.__name__.lower() if estimator is not None else None\n        self.task = task\n        self.metric = metric\n        self.metric_name = getattr(metric, \"__name__\", str(metric)) if metric is not None else None\n        self.cv = cv\n        self.cv_args = cv_args or {}\n        self.use_early_stopping = use_early_stopping\n        self.verbose = verbose\n        self.save = save\n        self.save_path = save_path\n        self.metric_precision = metric_precision\n        self.metric_threshold = metric_threshold\n        self.metric_args = metric_args or {}\n        self.estimators = []\n        self.fold_scores = []\n        self.oof_preds = None\n        self.overall_score = None\n        self.is_fitted = False\n\n    def _score(self, y_true, y_pred):\n        if self.metric is None:\n            return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))\n        return float(self.metric(y_true, y_pred, **self.metric_args))\n\n    def fit(self, X, y, fit_args=None):\n        fit_args = fit_args or {}\n        X_arr = X\n        y_arr = np.asarray(y)\n        self.oof_preds = np.zeros(len(y_arr), dtype=float)\n        groups = self.cv_args.get(\"groups\")\n        split_iter = self.cv.split(X_arr, y_arr, groups=groups) if self.cv is not None else [(np.arange(len(y_arr)), np.arange(len(y_arr)))]\n        self.estimators = []\n        self.fold_scores = []\n        for fold, (tr_idx, va_idx) in enumerate(split_iter):\n            model = clone(self.estimator)\n            kwargs = dict(fit_args)\n            # Keep only fit kwargs that broadly work; public artifact notebooks pass\n            # eval callbacks for LGB/CAT only when artifacts are absent. Ridge meta\n            # fit passes no kwargs. This compatibility shim is intentionally small.\n            try:\n                model.fit(X_arr.iloc[tr_idx] if hasattr(X_arr, 'iloc') else X_arr[tr_idx], y_arr[tr_idx], **kwargs)\n            except TypeError:\n                model.fit(X_arr.iloc[tr_idx] if hasattr(X_arr, 'iloc') else X_arr[tr_idx], y_arr[tr_idx])\n            pred = model.predict(X_arr.iloc[va_idx] if hasattr(X_arr, 'iloc') else X_arr[va_idx])\n            self.oof_preds[va_idx] = pred\n            self.estimators.append(model)\n            self.fold_scores.append(self._score(y_arr[va_idx], pred))\n        self.overall_score = self._score(y_arr, self.oof_preds)\n        self.is_fitted = True\n        if self.save and self.save_path:\n            os.makedirs(self.save_path, exist_ok=True)\n            joblib.dump(self, os.path.join(self.save_path, f\"{self.estimator_name}_trainer.pkl\"))\n        return self\n\n    def predict(self, X):\n        if not getattr(self, \"estimators\", None):\n            if hasattr(self, \"estimator\") and self.estimator is not None:\n                return self.estimator.predict(X)\n            raise AttributeError(\"Trainer has no estimators\")\n        return np.mean([m.predict(X) for m in self.estimators], axis=0)\n")
Path("ravaghi_features.py").write_text("\"\"\"Ravaghi public artifact feature builder for run_v7 submission.\n\nThis module intentionally preserves the public notebook's raw feature pipeline.\nThe c50-c54 OOF candidates were imported from Trainer.oof_preds, so inference\nmust call Trainer.predict on this same feature schema and use raw offset outputs.\n\"\"\"\nimport time\nimport multiprocessing\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom joblib import Parallel, delayed\nfrom numba import njit\nfrom scipy.signal import savgol_filter\nfrom scipy.spatial import cKDTree\n\n\nclass CFG:\n    dataset_path = None\n\n\nSEED=42\nNCPU=min(4,multiprocessing.cpu_count())\n\nFORMATIONS=[\"ANCC\",\"ASTNU\",\"ASTNL\",\"EGFDU\",\"EGFDL\",\"BUDA\"]\nPLANE_K=10; DENSE_SPW=60; DENSE_K=20; N_SPLITS=5\n\nBEAMS=[\n    (10,20.0,144.0,2,\"cons\"),\n    (10, 8.0, 64.0,2,\"loose\"),\n    ( 8,35.0,220.0,1,\"vcons\"),\n    (10,14.0, 90.0,5,\"sm5\"),\n    (20, 4.0, 36.0,3,\"vloose\"),\n    (12,12.0,100.0,3,\"mid\"),\n    (15,25.0,180.0,2,\"stiff\"),\n]\n\nPF_N=600; ANCC_N=600\nPF_MOM=0.993; PF_VN=0.005; PF_PN=0.01\nPF_GR_SIG_MIN=10.; PF_GR_SIG_MAX=60.; PF_GR_SIG_DEF=30.\nPF_INIT_V_STD=0.02; PF_INIT_SPR=0.5; PF_RESAMP=0.5\nPF_ROUGH_P=0.2; PF_ROUGH_V=0.003; PF_GR_WIN=5; PF_GR_WT=0.3\nANCC_ALPHA=0.998; ANCC_RN=0.002; ANCC_PN=0.005\nANCC_IR=0.01; ANCC_IS=0.3; ANCC_RP=0.1; ANCC_RR=0.001\n\n@njit(cache=True)\ndef _interp1(grid, v, vmin, step):\n    i = int((v - vmin) / step)\n    if i < 0: return grid[0]\n    n = len(grid) - 1\n    if i >= n: return grid[n]\n    t = (v - vmin) / step - i\n    return grid[i]*(1.-t) + grid[i+1]*t\n\n@njit(cache=True)\ndef _resamp(pos, aux, w, N, rp, rv):\n    cum = np.zeros(N+1)\n    for j in range(N): cum[j+1]=cum[j]+w[j]\n    u0=np.random.uniform(0.,1./N)\n    np2=np.empty(N); na=np.empty(N); ci=0\n    for j in range(N):\n        u=u0+j/N\n        while ci<N-1 and cum[ci+1]<u: ci+=1\n        np2[j]=pos[ci]+rp*np.random.randn()\n        na[j] =aux[ci]+rv*np.random.randn()\n    return np2,na\n\n@njit(cache=True)\ndef _beam_jit(sgr, tw_gr, si, BS, mc, es):\n    \"\"\"Beam search \u00b12 delta, Numba JIT.\"\"\"\n    n=len(sgr); nt=len(tw_gr); MAX=BS*6\n    bidx=np.zeros(BS,np.int64); bidx[0]=si\n    bcost=np.full(BS,1e30);     bcost[0]=0.; bn=np.int64(1)\n    hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)\n    cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)\n    for step in range(n):\n        gv=sgr[step]; nc=np.int64(0)\n        for bi in range(bn):\n            idx=bidx[bi]; cost=bcost[bi]\n            for d in range(-2,3):            # \u00b12: TVT can go down\n                ni=idx+d\n                if ni<0 or ni>=nt: continue\n                tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)\n                fnd=np.int64(-1)\n                for ci in range(nc):\n                    if cI[ci]==ni: fnd=ci; break\n                if fnd>=0:\n                    if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi\n                else:\n                    if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1\n        kept=min(BS,nc)\n        for i in range(kept):\n            mi=i\n            for j in range(i+1,nc):\n                if cC[j]<cC[mi]: mi=j\n            if mi!=i:\n                cI[i],cI[mi]=cI[mi],cI[i]\n                cC[i],cC[mi]=cC[mi],cC[i]\n                cP[i],cP[mi]=cP[mi],cP[i]\n        hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]\n        bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept\n    best=np.int64(0)\n    for b in range(1,bn):\n        if bcost[b]<bcost[best]: best=b\n    path=np.zeros(n,np.int64); b=best\n    for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]\n    return path\n\n@njit(cache=True)\ndef _pf_ancc(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,\n              ALPHA,RN,PN,IS,RP,RR,RESAMP):\n    pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N\n    for j in range(N):\n        pos[j]=ls+IS*np.random.randn()\n        rate[j]=ir+0.01*np.random.randn()\n    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.\n    for i in range(len(md_v)):\n        dm=md_v[i]-pm; dm=max(dm,1.)\n        for j in range(N):\n            rate[j]=ALPHA*rate[j]+RN*np.random.randn()\n            pos[j]+=rate[j]*dm+PN*np.random.randn()\n            tvt_j=pos[j]-z_v[i]\n            tvt_j=max(tvt_j,vmin-50.); tvt_j=min(tvt_j,vmin+len(gg)*step+50.)\n            pos[j]=tvt_j+z_v[i]\n        if not np.isnan(gr_v[i]):\n            ws=0.\n            for j in range(N):\n                eg=_interp1(gg,pos[j]-z_v[i],vmin,step)\n                d=(gr_v[i]-eg)/gs\n                lk=max(np.exp(-0.5*d*d) if d*d<600. else 0.,1e-300)\n                w[j]*=lk; ws+=w[j]\n            if ws>0.:\n                for j in range(N): w[j]/=ws\n            else:\n                for j in range(N): w[j]=1./N\n        ne=0.\n        for j in range(N): ne+=w[j]*w[j]\n        if 1./ne<RESAMP*N:\n            pos,rate=_resamp(pos,rate,w,N,RP,RR)\n            for j in range(N): w[j]=1./N\n        tv=0.\n        for j in range(N): tv+=w[j]*(pos[j]-z_v[i])\n        pts[i]=tv; va=0.\n        for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2\n        std_[i]=va**0.5; pm=md_v[i]\n    return pts,std_\n\n@njit(cache=True)\ndef _pf_z(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,\n          gs,ip,iv,beta,icpt,zsig,N,\n          MOM,VN,PN,GR_WT,RP,RV,RESAMP):\n    pos=np.empty(N); vel=np.empty(N); w=np.ones(N)/N\n    for j in range(N):\n        pos[j]=ip+0.5*np.random.randn()\n        vel[j]=iv+0.02*np.random.randn()\n    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.; pz=z_v[0]-1.\n    for i in range(len(md_v)):\n        dm=md_v[i]-pm; dm=max(dm,1.)\n        dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt\n        for j in range(N):\n            vel[j]=MOM*vel[j]+VN*np.random.randn()\n            pos[j]+=vel[j]*dm+PN*np.random.randn()\n            pos[j]=max(pos[j],vmin-50.); pos[j]=min(pos[j],vmin+len(gg_p)*step+50.)\n        if not np.isnan(gr_v[i]):\n            ws=0.\n            for j in range(N):\n                ep=_interp1(gg_p,pos[j],vmin,step)\n                dp=(gr_v[i]-ep)/gs\n                lp=max(np.exp(-0.5*dp*dp) if dp*dp<600. else 0.,1e-300)\n                if not np.isnan(gr_sm_v[i]):\n                    es=_interp1(gg_s,pos[j],vmin,step)\n                    ds=(gr_sm_v[i]-es)/(gs*1.5)\n                    ls=max(np.exp(-0.5*ds*ds) if ds*ds<600. else 0.,1e-300)\n                    lk=(1.-GR_WT)*lp+GR_WT*ls\n                else: lk=lp\n                lk=max(lk,1e-300); w[j]*=lk; ws+=w[j]\n            if ws>0.:\n                for j in range(N): w[j]/=ws\n            else:\n                for j in range(N): w[j]=1./N\n        ws2=0.\n        for j in range(N):\n            dv=(vel[j]-ve)/max(zsig*2.,0.005)\n            lz=max(np.exp(-0.5*dv*dv) if dv*dv<600. else 0.,1e-300)\n            w[j]*=lz; ws2+=w[j]\n        if ws2>0.:\n            for j in range(N): w[j]/=ws2\n        else:\n            for j in range(N): w[j]=1./N\n        ne=0.\n        for j in range(N): ne+=w[j]*w[j]\n        if 1./ne<RESAMP*N:\n            pos,vel=_resamp(pos,vel,w,N,RP,RV)\n            for j in range(N): w[j]=1./N\n        wm=0.\n        for j in range(N): wm+=w[j]*pos[j]\n        pts[i]=wm; va=0.\n        for j in range(N): va+=w[j]*(pos[j]-wm)**2\n        std_[i]=va**0.5; pm=md_v[i]; pz=z_v[i]\n    return pts,std_\n\n# Dense grid for O(1) typewell lookup\ndef _grid(tw_tvt,tw_gr,step=0.2):\n    tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max())\n    tvt_g=np.arange(tmin,tmax+step,step)\n    return np.interp(tvt_g,tw_tvt,tw_gr).astype(np.float64),float(tmin),float(step)\n\ndef _gr_sig(hw,tw_tvt,tw_gr):\n    kn=hw[hw['TVT_input'].notna()&hw['GR'].notna()]\n    if len(kn)<20: return float(PF_GR_SIG_DEF)\n    return float(np.clip(np.std(kn['GR'].values-np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)),\n                          PF_GR_SIG_MIN,PF_GR_SIG_MAX))\n\ndef _nn(arr,v):\n    i=int(np.searchsorted(arr,v,'left'))\n    if i>=len(arr): return len(arr)-1\n    if i>0 and abs(arr[i-1]-v)<=abs(arr[i]-v): return i-1\n    return i\n\ndef _smooth(vals,fb,r):\n    s=pd.Series(vals,dtype='float32').interpolate(limit_direction='both').fillna(fb)\n    return (s.rolling(r*2+1,center=True,min_periods=1).mean() if r>0 else s).to_numpy(np.float32)\n\ndef beam_search(gr_h,tw_tvt,tw_gr,start_tvt,bs,mc,es,r):\n    si=_nn(tw_tvt,start_tvt)\n    sgr=_smooth(gr_h,float(np.nanmean(tw_gr)),r).astype(np.float64)\n    path=_beam_jit(sgr,tw_gr.astype(np.float64),si,bs,float(mc),float(es))\n    return tw_tvt[path].astype(np.float32)\n\ndef run_pf_ancc(hw,tw_tvt,tw_gr,N=ANCC_N):\n    gs=_gr_sig(hw,tw_tvt,tw_gr)\n    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]\n    if len(ev)==0: return np.array([]),np.array([])\n    ls=float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])\n    tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values)\n    dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0\n    ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.\n    gg,gmin,gst=_grid(tw_tvt,tw_gr)\n    pts,std=_pf_ancc(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),\n                      ev['GR'].values.astype(np.float64),gg,gmin,gst,\n                      gs,ls,ir,N,ANCC_ALPHA,ANCC_RN,ANCC_PN,ANCC_IS,ANCC_RP,ANCC_RR,PF_RESAMP)\n    return pts.astype(np.float32),std.astype(np.float32)\n\ndef run_pf_z(hw,tw_tvt,tw_gr,N=PF_N):\n    gs=_gr_sig(hw,tw_tvt,tw_gr)\n    tw_s=pd.Series(tw_gr).rolling(PF_GR_WIN,center=True,min_periods=1).mean().values.astype(np.float32)\n    kna=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]\n    if len(ev)==0: return np.array([]),np.array([])\n    dz_k=np.diff(kna['Z'].values); dvt=np.diff(kna['TVT_input'].values)\n    dmd_k=np.diff(kna['MD'].values); m2=dmd_k>0\n    if m2.sum()>=10:\n        vz=dz_k[m2]/dmd_k[m2]; vt=dvt[m2]/dmd_k[m2]\n        A=np.column_stack([vz,np.ones_like(vz)]); c,_,_,_=np.linalg.lstsq(A,vt,rcond=None)\n        beta,icpt,zsig=float(c[0]),float(c[1]),max(float(np.std(vt-(c[0]*vz+c[1]))),0.001)\n    else: beta,icpt,zsig=-1.,0.,0.1\n    t2=kna.tail(20); dvt2=np.diff(t2['TVT_input'].values); dmd2=np.diff(t2['MD'].values); m3=dmd2>0\n    iv=float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum()>=3 else 0.\n    gg,gmin,gst=_grid(tw_tvt,tw_gr)\n    gs2,_,_=_grid(tw_tvt,tw_s)\n    gr_sm=hw['GR'].rolling(PF_GR_WIN,center=True,min_periods=1).mean()\n    pts,std=_pf_z(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),\n                   ev['GR'].values.astype(np.float64),\n                   gr_sm.loc[ev.index].values.astype(np.float64),\n                   gg,gs2,gmin,gst,gs,float(kna['TVT_input'].iloc[-1]),iv,\n                   beta,icpt,zsig,N,\n                   PF_MOM,PF_VN,PF_PN,PF_GR_WT,PF_ROUGH_P,PF_ROUGH_V,PF_RESAMP)\n    return pts.astype(np.float32),std.astype(np.float32)\n\n\n_md=np.linspace(1,50,20,np.float64); _z=np.zeros(20,np.float64); _gr=np.full(20,50.,np.float64)\n_gg=np.linspace(45,55,100,np.float64)\n_pf_ancc(_md,_z,_gr,_gg,45.,0.1,20.,50.,0.,8,0.998,0.002,0.005,0.3,0.1,0.001,0.5)\n_pf_z(_md,_z,_gr,_gr,_gg,_gg,45.,0.1,20.,50.,0.,-1.,0.,0.1,8,0.993,0.005,0.01,0.3,0.2,0.003,0.5)\n_beam_jit(np.random.randn(30),np.random.randn(50),25,8,15.,100.)\n\ndef robust_slope(x,y,w=None):\n    x=np.asarray(x,float); y=np.asarray(y,float)\n    m=np.isfinite(x)&np.isfinite(y)\n    if m.sum()<2 or np.std(x[m])<1e-6: return 0.\n    return float(np.polyfit(x[m],y[m],1)[0])\n\ndef affine_cal(kgr,tw_at_k,min_pts=20):\n    v=np.isfinite(kgr)&np.isfinite(tw_at_k)\n    if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6:\n        return 1.,float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.\n    a,b=np.polyfit(tw_at_k[v],kgr[v],1); return float(a),float(b)\n\ndef seg_b_well(ktvt,kz,form_col):\n    \"\"\"Segment b_well: early/mid/late thirds + full prefix.\n    Returns (b_full, b_early, b_mid, b_late, b_wls) for feature richness.\"\"\"\n    bv=ktvt+kz-form_col; n=len(bv)\n    b_full=float(np.median(bv))\n    b_late=float(np.median(bv[max(0,n-50):])) if n>=5 else b_full\n    t1,t2=n//3, 2*n//3\n    b_early=float(np.median(bv[:max(1,t1)])) if t1>0 else b_full\n    b_mid  =float(np.median(bv[t1:max(t1+1,t2)])) if t2>t1 else b_full\n    # WLS (tail-upweighted)\n    w=np.exp(0.02*np.arange(n)); w/=w.sum()\n    b_wls=float(np.dot(w,bv))\n    return b_full,b_early,b_mid,b_late,b_wls\n\ndef multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3):\n    \"\"\"Multi-scale NCC. Returns score-weighted ensemble + per-scale signals.\"\"\"\n    out=[]\n    for hw in hws:\n        win=2*hw+1; nk=len(kgr); nh=len(hgr)\n        if nk<win+1 or nh==0:\n            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue\n        kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)\n        hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)\n        sts=np.arange(0,nk-win+1,stride,dtype=np.int32); M=len(sts)\n        if M==0:\n            out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue\n        C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)\n        Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)\n        hp=np.pad(hg,hw,mode='edge')\n        H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)\n        Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)\n        ncc=Hn@Cn.T/win; best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)\n        out.append((ktvt[np.clip(sts[best]+hw,0,nk-1)].astype(np.float32),score))\n    # Score-weighted ensemble (NEW: softmax-weighted combination)\n    tvts=np.stack([o[0] for o in out],1); scores=np.stack([o[1] for o in out],1)\n    sw=np.exp(3.*scores); sw/=sw.sum(1,keepdims=True)+1e-9\n    sc_ens=(tvts*sw).sum(1).astype(np.float32)\n    return out, sc_ens   # [(tvt8,sc8),(tvt15,sc15),(tvt25,sc25)], ensemble\n\nclass FormationPlaneKNN:\n    def __init__(self,well_ids,data_dir):\n        rows=[]\n        for wid in well_ids:\n            p=data_dir/f'{wid}__horizontal_well.csv'\n            try: df=pd.read_csv(p,usecols=['X','Y']+FORMATIONS).dropna()\n            except: continue\n            if len(df)==0: continue\n            row={'wid':wid,'x':float(df['X'].median()),'y':float(df['Y'].median())}\n            for c in FORMATIONS: row[f'{c}_m']=float(df[c].median())\n            rows.append(row)\n        self.df=pd.DataFrame(rows); self.wmap={w:i for i,w in enumerate(self.df['wid'])}\n        xy=self.df[['x','y']].to_numpy(); self.scale=np.where(xy.std(0)<1e-3,1.,xy.std(0))\n        self.tree=cKDTree(xy/self.scale)\n        self.xa=self.df['x'].to_numpy(); self.ya=self.df['y'].to_numpy()\n        self.fa=self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)\n\n    def impute(self,xy_q,self_wid=None,k=PLANE_K):\n        q=xy_q/self.scale; nf=min(k+5,len(self.df))\n        dist,idx=self.tree.query(q,k=nf,workers=-1)\n        if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)\n        ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]\n        dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)\n        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.).astype(np.float64)\n        xn=self.xa[ik]; yn=self.ya[ik]; fn=self.fa[ik]; wx=w*xn; wy=w*yn\n        A=np.zeros((len(q),3,3))\n        A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)\n        A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)\n        A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)\n        A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9\n        rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(w[:,:,None]*fn).sum(1)],1)\n        try: coef=np.linalg.solve(A,rhs)\n        except:\n            coef=np.zeros((len(q),3,6))\n            for r in range(len(q)):\n                try: coef[r]=np.linalg.pinv(A[r])@rhs[r]\n                except: pass\n        Xq=xy_q[:,0]; Yq=xy_q[:,1]\n        pred=(Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)\n        pred[~vk.any(1)]=self.fa.mean(0)\n        return pred,np.where(vk,dk,np.inf).min(1).astype(np.float32)\n\nclass DenseANCCImputer:\n    def __init__(self,well_ids,data_dir,spw=DENSE_SPW):\n        xs,ys,anccs,wids=[],[],[],[]\n        for wid in well_ids:\n            p=data_dir/f'{wid}__horizontal_well.csv'\n            try: df=pd.read_csv(p,usecols=['X','Y','ANCC']).dropna()\n            except: continue\n            if len(df)==0: continue\n            ix=np.linspace(0,len(df)-1,min(spw,len(df)),dtype=int); s=df.iloc[ix]\n            xs.append(s['X'].values); ys.append(s['Y'].values)\n            anccs.append(s['ANCC'].values); wids.extend([wid]*len(s))\n        self.xy=np.column_stack([np.concatenate(xs),np.concatenate(ys)])\n        self.ancc=np.concatenate(anccs).astype(np.float32); self.wids=np.array(wids)\n        self.scale=np.where(self.xy.std(0)<1e-3,1.,self.xy.std(0))\n        self.tree=cKDTree(self.xy/self.scale)\n\n    def impute(self,xy_q,self_wid=None,k=DENSE_K,nfetch=5000):\n        xy_q=np.atleast_2d(xy_q); q=xy_q/self.scale; nf=min(nfetch,len(self.ancc))\n        dist,idx=self.tree.query(q,k=nf,workers=-1)\n        if self_wid: dist=np.where(self.wids[idx]==self_wid,np.inf,dist)\n        ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]\n        dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)\n        vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.)\n        sw=w.sum(1); safe=np.where(sw<1e-9,1.,sw); an=self.ancc[ik]\n        ap=(an*w).sum(1)/safe; ap=np.where(sw<1e-9,float(self.ancc.mean()),ap)\n        var=((an-ap[:,None])**2*w).sum(1)/safe\n        return ap.astype(np.float32),np.sqrt(np.maximum(var,0.)).astype(np.float32),np.where(vk,dk,np.inf).min(1).astype(np.float32)\n\n_FI=None\n_DI=None\nANCH_OFFS=np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80],np.float32)\nBEAM_OFFS=np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40],np.float32)\nSC_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)\nPF_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)\n\ndef build_well(hw_path,tw_path,is_train):\n    global _FI,_DI\n    wid=Path(hw_path).stem.replace('__horizontal_well','')\n    try:\n        hw=pd.read_csv(hw_path); tw=pd.read_csv(tw_path).sort_values('TVT')\n    except: return None\n    if is_train and 'TVT' not in hw.columns: return None\n    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]\n    if len(ev)==0 or len(kn)<10: return None\n    if is_train and hw['TVT'].isna().all(): return None\n    tw_tvt=tw['TVT'].to_numpy(np.float32); tw_gr=tw['GR'].to_numpy(np.float32)\n    if len(tw_tvt)<3: return None\n\n    pf_a,std_a=run_pf_ancc(hw,tw_tvt,tw_gr)\n    if len(pf_a)==0: return None\n    pf_z,std_z=run_pf_z(hw,tw_tvt,tw_gr)\n    pf_use=pf_a.astype(np.float32); std_use=std_a.astype(np.float32)\n    has_z=len(pf_z)==len(pf_a) and not np.any(np.isnan(pf_z))\n\n    lk=kn.iloc[-1]; last_tvt=float(lk['TVT_input'])\n    gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))\n    hgr=gr_full.iloc[ev.index[0]:].to_numpy(np.float32)\n    kgr=gr_full.iloc[:len(kn)].to_numpy(np.float32)\n\n    # 7 beams (Numba JIT \u00b12)\n    bpaths={}\n    for (bs,mc,es,r,tag) in BEAMS:\n        bpaths[tag]=beam_search(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r)\n    beam_ref=(bpaths['cons']+bpaths['sm5'])/2.\n\n    # Multi-scale NCC \u2192 score-weighted ensemble\n    ktvt=kn['TVT_input'].to_numpy(np.float32)\n    sc_res,sc_ens=multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3)\n    sc8,sc8s=sc_res[0]; sc15,sc15s=sc_res[1]; sc25,sc25s=sc_res[2]\n    sc_cons=(sc8+sc15+sc25)/3.\n    sc_trust=float(np.clip(len(kn)/200.,0.,0.6))\n    hyb_ref=(1-sc_trust)*beam_ref+sc_trust*sc_ens  # use ensemble not single\n\n    tw_at_k=np.interp(ktvt,tw_tvt,tw_gr).astype(np.float32)\n    a_cal,b_cal=affine_cal(kgr,tw_at_k)\n    kmd=kn['MD'].to_numpy(np.float32); kz=kn['Z'].to_numpy(np.float32)\n    pfx_rmse=float(np.sqrt(np.mean((kgr-tw_at_k)**2)))\n    slp_all=robust_slope(kmd,ktvt); slp_50=robust_slope(kmd[-50:],ktvt[-50:])\n    slp_z=robust_slope(kz,ktvt)\n\n    swid=wid if is_train else None\n    xy_ev=ev[['X','Y']].to_numpy(np.float64); xy_kn=kn[['X','Y']].to_numpy(np.float64)\n    form_ev,knn_d=_FI.impute(xy_ev,self_wid=swid)\n    form_kn,_   =_FI.impute(xy_kn,self_wid=swid)\n    z_kn=kn['Z'].to_numpy(np.float32); z_ev=ev['Z'].to_numpy(np.float32)\n\n    # Per-formation: segment b_well (early/mid/late/wls) + TVT + known-zone RMSE\n    tvt_fs={}; form_rmse={}; form_list=[]\n    for fi2,fn in enumerate(FORMATIONS):\n        b_full,b_early,b_mid,b_late,b_wls=seg_b_well(ktvt,z_kn,form_kn[:,fi2])\n        tvt_f  =(-z_ev+form_ev[:,fi2]+b_full ).astype(np.float32)\n        tvt_fw =(-z_ev+form_ev[:,fi2]+b_wls  ).astype(np.float32)\n        tvt_f50=(-z_ev+form_ev[:,fi2]+b_late ).astype(np.float32)\n        tvt_fs[f'tvtF_{fn}']=tvt_f; tvt_fs[f'tvtFw_{fn}']=tvt_fw\n        tvt_fs[f'tvtF50_{fn}']=tvt_f50\n        tvt_fs[f'bw_{fn}']=np.float32(b_full); tvt_fs[f'bww_{fn}']=np.float32(b_wls)\n        tvt_fs[f'bw50_{fn}']=np.float32(b_late)\n        tvt_fs[f'bw_early_{fn}']=np.float32(b_early)   # NEW: early segment\n        tvt_fs[f'bw_mid_{fn}']=np.float32(b_mid)       # NEW: mid segment\n        form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_full))**2)))\n        form_list.append(tvt_f)\n\n    fs=np.stack(form_list,1)\n    form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32)\n    form_std_d =fs.std(1).astype(np.float32)\n    form_rng_d =(fs.max(1)-fs.min(1)).astype(np.float32)\n\n    d_ancc,d_std,d_dist=_DI.impute(xy_ev,self_wid=swid)\n    d_kn,d_std_kn,_=_DI.impute(xy_kn,self_wid=swid)\n    b_vd=ktvt+z_kn-d_kn\n    _,b_de,b_dm,b_dl,b_dw=seg_b_well(ktvt,z_kn,d_kn)\n    b_d=float(np.median(b_vd))\n    tvt_dense  =(-z_ev+d_ancc+b_d  ).astype(np.float32)\n    tvt_densew =(-z_ev+d_ancc+b_dw ).astype(np.float32)\n    tvt_dense50=(-z_ev+d_ancc+b_dl ).astype(np.float32)\n    res_kn=ktvt+z_kn-d_kn\n    d_rmse=float(np.sqrt(np.mean(res_kn**2))); d_bias=float(np.mean(res_kn)); d_nb_std=float(np.mean(d_std_kn))\n\n    all_sigs=[pf_use]+[p for p in bpaths.values()]+[sc8,sc15,sc25,sc_ens,tvt_fs['tvtF_ANCC'],tvt_dense]\n    sig_mat=np.stack(all_sigs,1)\n    sig_std=sig_mat.std(1).astype(np.float32)\n    sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)\n\n    gr_s=pd.Series(gr_full.values); rolls={}\n    for w in [5,21,51,101]:\n        r=gr_s.rolling(w,center=True,min_periods=1)\n        rolls[f'grm{w}']=r.mean().iloc[ev.index].values.astype(np.float32)\n        rolls[f'grs{w}']=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)\n    for lag in [1,5,15,30]:\n        rolls[f'glag{lag}']=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)\n        rolls[f'glead{lag}']=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)\n    gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)\n    gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)\n    gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)\n    gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)\n                   ).iloc[ev.index].values.astype(np.float32)\n\n    hmd=ev['MD'].to_numpy(np.float32); md_since=hmd-float(lk['MD'])\n    slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32)\n    slp_b_50 =(last_tvt+slp_50 *md_since).astype(np.float32)\n\n    mdd=hw['MD'].diff().replace(0,np.nan)\n    dzdmd=(hw['Z'].diff()/mdd).iloc[ev.index].values.astype(np.float32)\n    dxdmd=(hw['X'].diff()/mdd).iloc[ev.index].values.astype(np.float32)\n    dydmd=(hw['Y'].diff()/mdd).iloc[ev.index].values.astype(np.float32)\n\n    nh=len(ev); frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)\n    def sc(v): return np.full(nh,np.float32(v),np.float32)\n\n    feats={\n        'well':wid,'id':[f'{wid}_{i}' for i in ev.index],\n        'last_known_tvt':sc(last_tvt),\n        'pf_ancc':pf_use,'pf_ancc_std':std_use,\n        'pf_ancc_delta':(pf_use-last_tvt).astype(np.float32),\n        'pf_z':(pf_z.astype(np.float32) if has_z else sc(last_tvt)),\n        'pf_z_delta':((pf_z-last_tvt).astype(np.float32) if has_z else sc(0.)),\n        'pf_vs_z':((pf_use-pf_z.astype(np.float32)) if has_z else sc(0.)),\n        **{f'beam_{t}_d':(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},\n        'beam_mean_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),\n        'beam_std_d': np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),\n        'beam_med_d': np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),\n        'sc8_d':(sc8-np.float32(last_tvt)).astype(np.float32),'sc8_sc':sc8s,\n        'sc15_d':(sc15-np.float32(last_tvt)).astype(np.float32),'sc15_sc':sc15s,\n        'sc25_d':(sc25-np.float32(last_tvt)).astype(np.float32),'sc25_sc':sc25s,\n        'sc_cons_d':(sc_cons-np.float32(last_tvt)).astype(np.float32),\n        'sc_ens_d':(sc_ens-np.float32(last_tvt)).astype(np.float32),  # score-weighted ensemble\n        'sc_trust':sc(sc_trust),'hyb_d':(hyb_ref-np.float32(last_tvt)).astype(np.float32),\n        'sig_std':sig_std,'sig_mean_d':sig_mean,\n        **tvt_fs,\n        **{f'frm_rmse_{fn}':sc(form_rmse[fn]) for fn in FORMATIONS},\n        'form_mean_d':form_mean_d,'form_std_d':form_std_d,'form_rng_d':form_rng_d,\n        'spatial_ancc_d':(form_ev[:,0]-np.float32(np.interp(last_tvt,tw_tvt,tw_gr))),\n        'spatial_knn_dist':knn_d,\n        'dense_ancc':d_ancc,'dense_std':d_std,'dense_dist':d_dist,\n        'tvt_dense_d' :(tvt_dense -last_tvt).astype(np.float32),\n        'tvt_densew_d':(tvt_densew-last_tvt).astype(np.float32),\n        'tvt_dense50_d':(tvt_dense50-last_tvt).astype(np.float32),\n        'dense_rmse':sc(d_rmse),'dense_bias':sc(d_bias),'dense_nb_std':sc(d_nb_std),\n        'pf_vs_spatial':(pf_use-tvt_fs['tvtF_ANCC']).astype(np.float32),\n        'pf_vs_dense':(pf_use-tvt_dense).astype(np.float32),\n        'spatial_vs_dense':(tvt_fs['tvtF_ANCC']-tvt_dense).astype(np.float32),\n        'beam_vs_spatial':(bpaths['cons']-tvt_fs['tvtF_ANCC']).astype(np.float32),\n        'sc_vs_beam':(sc_ens-bpaths['cons']).astype(np.float32),\n        'cal_a':sc(a_cal),'cal_b':sc(b_cal),\n        'pfx_rmse':sc(pfx_rmse),'known_len':sc(len(kn)),'eval_len':sc(nh),\n        'slp_all':sc(slp_all),'slp_50':sc(slp_50),'slp_z':sc(slp_z),\n        'slp_b_d_all':(slp_b_all-last_tvt).astype(np.float32),\n        'slp_b_d_50': (slp_b_50 -last_tvt).astype(np.float32),\n        'ktvt_range':sc(float(np.ptp(ktvt))),'ktvt_std':sc(float(ktvt.std())),\n        'md_since':md_since,'frac':frac,'frac2':frac**2,'sqrt_frac':np.sqrt(frac),\n        'z':z_ev,\n        'dx':(ev['X']-float(lk['X'])).to_numpy(np.float32),\n        'dy':(ev['Y']-float(lk['Y'])).to_numpy(np.float32),\n        'dz':(z_ev-float(lk['Z'])).astype(np.float32),\n        'dxy':np.sqrt((ev['X']-float(lk['X']))**2+(ev['Y']-float(lk['Y']))**2).to_numpy(np.float32),\n        'dzdmd':dzdmd,'dxdmd':dxdmd,'dydmd':dydmd,\n        'gr':hgr,'gr_d1':gr_d1,'gr_d2':gr_d2,'gr_env':gr_env,'gr_nrg':gr_nrg,\n        'gr_vs_tw_anc':hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),\n        'gr_vs_slp_all':hgr-np.interp(slp_b_all,tw_tvt,tw_gr).astype(np.float32),\n        **{f'tda{int(o)}' :hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr)) for o in ANCH_OFFS},\n        **{f'tdbc{int(o)}':hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32) for o in BEAM_OFFS},\n        **{f'tdsc{int(o)}':hgr-np.interp(sc_ens+o,tw_tvt,tw_gr).astype(np.float32) for o in SC_OFFS},\n        **{f'tdpf{int(o)}':hgr-np.interp(pf_use+o,tw_tvt,tw_gr).astype(np.float32) for o in PF_OFFS},\n        'tw_range':sc(float(np.ptp(tw_tvt))),'tw_gr_mean':sc(float(tw_gr.mean())),\n    }\n    for k,v in rolls.items(): feats[k]=v\n    result=pd.DataFrame(feats)\n    if is_train:\n        if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None\n        result['target']=(ev['TVT'].to_numpy(np.float32)-np.float32(last_tvt))\n    return result\n\ndef build_dataset(paths,is_train,label):\n    args=[(str(p),str(p.parent/f'{p.stem.replace(\"__horizontal_well\",\"\")}__typewell.csv'),is_train)\n          for p in paths\n          if (p.parent/f'{p.stem.replace(\"__horizontal_well\",\"\")}__typewell.csv').exists()]\n    t0=time.time()\n    res=Parallel(n_jobs=NCPU,prefer='threads',verbose=3)(\n        delayed(build_well)(hp,tp,it) for hp,tp,it in args)\n    parts=[r for r in res if r is not None]\n    return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()\n\n\ndef build_test_features(dataset_path):\n    \"\"\"Build ravaghi-schema features for competition test lateral rows.\"\"\"\n    global _FI, _DI\n    CFG.dataset_path = Path(dataset_path)\n    hw_paths = sorted((CFG.dataset_path / \"train\").glob(\"*__horizontal_well.csv\"))\n    train_wids = [p.stem.replace(\"__horizontal_well\", \"\") for p in hw_paths]\n    _FI = FormationPlaneKNN(train_wids, CFG.dataset_path / \"train\")\n    _DI = DenseANCCImputer(train_wids, CFG.dataset_path / \"train\")\n    test_paths = sorted((CFG.dataset_path / \"test\").glob(\"*__horizontal_well.csv\"))\n    return build_dataset(test_paths, is_train=False, label=\"test\")\n")
print("helper modules written")

In [ ]:
"""ROGII inference-only kernel — loads pre-trained models and runs prediction.

Does NOT train any models! Loads from dataset:
- lgb_model.txt (LightGBM native format)
- cat_model.cbm (CatBoost native format)
- feat_cols.pkl (pickled list of feature columns)
"""

In [ ]:
import os, time, warnings, pickle, json
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from numba import njit
import joblib
import lightgbm as lgb
from catboost import CatBoostRegressor

# Discover Kaggle input path (varies by competition: some at /kaggle/input/<slug>,
# others at /kaggle/input/competitions/<slug>).
def _find_kaggle_input(slug="rogii-wellbore-geology-prediction"):
    base = f"/kaggle/input/competitions/{slug}"
    if os.path.isdir(base): return base
    base = f"/kaggle/input/{slug}"
    if os.path.isdir(base): return base
    return None  # not on Kaggle

INPUT_DIR = _find_kaggle_input() or "/Users/liucong/code/kaggle/ROGII/rogii-wellbore-geology-prediction"
TRAIN_DIR = f"{INPUT_DIR}/train"
TEST_DIR  = f"{INPUT_DIR}/test"
OUT_PATH  = "/kaggle/working/submission.csv" if _find_kaggle_input() else "/Users/liucong/code/kaggle/ROGII/results/round_010/submission_inference.csv"

def _find_dataset_input(*slugs, local=None):
    for slug in slugs:
        for base in (
            f"/kaggle/input/{slug}",
            f"/kaggle/input/datasets/{slug}",
        ):
            if os.path.isdir(base):
                return base
    if local and os.path.isdir(local):
        return local
    return None

RAVAGHI_DIR = _find_dataset_input(
    "wellbore-geology-prediction-artifacts",
    "ravaghi/wellbore-geology-prediction-artifacts",
    local="/Users/liucong/code/kaggle/ROGII/experiments/public_resources/datasets_raw/ravaghi-artifacts",
)
V10_DIR = _find_dataset_input(
    "rogii-v10-fresh-artifacts",
    "thbdh5765/rogii-v10-fresh-artifacts",
    local="/Users/liucong/code/kaggle/ROGII/experiments/public_resources/datasets_raw/thbdh5765_rogii-v10-fresh-artifacts",
)

print(f"INPUT_DIR = {INPUT_DIR}")
print(f"RAVAGHI_DIR = {RAVAGHI_DIR}")
print(f"V10_DIR = {V10_DIR}")
assert os.path.isdir(TRAIN_DIR), f"TRAIN_DIR missing: {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR),  f"TEST_DIR  missing: {TEST_DIR}"
assert RAVAGHI_DIR and os.path.isdir(RAVAGHI_DIR), "RAVAGHI_DIR missing"
assert V10_DIR and os.path.isdir(V10_DIR), "V10_DIR missing"

ROLLING_WINS = [5, 21, 51, 101]

RUN_V7_WEIGHTS = {
    "c20_r9_pf128_full": 0.5028569013483001,
    "v10_tabicl_A": 0.19441129734773116,
    "c54_ravaghi_cat2": 0.09492358689965985,
    "c50_ravaghi_lgb1": 0.060044610924396336,
    "c52_ravaghi_lgb3": 0.04821063418364803,
    "c51_ravaghi_lgb2": 0.04564549095942647,
    "c53_ravaghi_cat1": 0.016237222462336894,
    "v10_tabicl_B": 0.01279830211826902,
}
RAVAGHI_MODELS = {
    "c50_ravaghi_lgb1": "models/lightgbm-1/lgbmregressor_trainer_20260526182612.pkl",
    "c51_ravaghi_lgb2": "models/lightgbm-2/lgbmregressor_trainer_20260526190415.pkl",
    "c52_ravaghi_lgb3": "models/lightgbm-3/lgbmregressor_trainer_20260526192806.pkl",
    "c53_ravaghi_cat1": "models/catboost-1/catboostregressor_trainer_20260526193740.pkl",
    "c54_ravaghi_cat2": "models/catboost-2/catboostregressor_trainer_20260526194838.pkl",
}
V10_KEYS = ["lgb123", "lgb42", "lgb7", "cb42", "cb7", "cb123", "tabicl_A", "tabicl_B"]


# ── PF (numba JIT, verbatim from LB-7.776 kernel) ─────────────────────────────
PF_N=600; ANCC_N=600
PF_MOM=0.993; PF_VN=0.005; PF_PN=0.01
PF_GR_SIG_MIN=10.; PF_GR_SIG_MAX=60.; PF_GR_SIG_DEF=30.
PF_INIT_V_STD=0.02; PF_INIT_SPR=0.5; PF_RESAMP=0.5
PF_ROUGH_P=0.2; PF_ROUGH_V=0.003; PF_GR_WIN=5; PF_GR_WT=0.3
ANCC_ALPHA=0.998; ANCC_RN=0.002; ANCC_PN=0.005
ANCC_IR=0.01; ANCC_IS=4.5; ANCC_RP=0.1; ANCC_RR=0.001

# Ensemble PF tracks total log-likelihood + uses sp45 init spread (kernel pattern)
ENS_N_SEEDS = 128   # hill climb v3: PF 128-seed is the dominant ensemble member
ENS_SCALES  = [3.0, 5.0, 8.0, 12.0]
ENS_ANCC_IS = 4.5   # sp45 patch (kernel sel15-vb-best)


@njit(cache=True)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i]*(1.-t) + grid[i+1]*t


@njit(cache=True)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N+1)
    for j in range(N): cum[j+1]=cum[j]+w[j]
    u0=np.random.uniform(0.,1./N)
    np2=np.empty(N); na=np.empty(N); ci=0
    for j in range(N):
        u=u0+j/N
        while ci<N-1 and cum[ci+1]<u: ci+=1
        np2[j]=pos[ci]+rp*np.random.randn()
        na[j] =aux[ci]+rv*np.random.randn()
    return np2,na


@njit(cache=True)
def _pf_ancc(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,ALPHA,RN,PN,IS,RP,RR,RESAMP):
    pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ls+IS*np.random.randn()
        rate[j]=ir+0.01*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        for j in range(N):
            rate[j]=ALPHA*rate[j]+RN*np.random.randn()
            pos[j]+=rate[j]*dm+PN*np.random.randn()
            tvt_j=pos[j]-z_v[i]
            tvt_j=max(tvt_j,vmin-50.); tvt_j=min(tvt_j,vmin+len(gg)*step+50.)
            pos[j]=tvt_j+z_v[i]
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                d=(gr_v[i]-eg)/gs
                lk=max(np.exp(-0.5*d*d) if d*d<600. else 0.,1e-300)
                w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,rate=_resamp(pos,rate,w,N,RP,RR)
            for j in range(N): w[j]=1./N
        tv=0.
        for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
        pts[i]=tv; va=0.
        for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2
        std_[i]=va**0.5; pm=md_v[i]
    return pts,std_


@njit(cache=True)
def _pf_z(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,gs,ip,iv,beta,icpt,zsig,N,
          MOM,VN,PN,GR_WT,RP,RV,RESAMP):
    pos=np.empty(N); vel=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ip+0.5*np.random.randn()
        vel[j]=iv+0.02*np.random.randn()
    pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.; pz=z_v[0]-1.
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt
        for j in range(N):
            vel[j]=MOM*vel[j]+VN*np.random.randn()
            pos[j]+=vel[j]*dm+PN*np.random.randn()
            pos[j]=max(pos[j],vmin-50.); pos[j]=min(pos[j],vmin+len(gg_p)*step+50.)
        if not np.isnan(gr_v[i]):
            ws=0.
            for j in range(N):
                ep=_interp1(gg_p,pos[j],vmin,step)
                dp=(gr_v[i]-ep)/gs
                lp=max(np.exp(-0.5*dp*dp) if dp*dp<600. else 0.,1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es=_interp1(gg_s,pos[j],vmin,step)
                    ds=(gr_sm_v[i]-es)/(gs*1.5)
                    ls=max(np.exp(-0.5*ds*ds) if ds*ds<600. else 0.,1e-300)
                    lk=(1.-GR_WT)*lp+GR_WT*ls
                else: lk=lp
                lk=max(lk,1e-300); w[j]*=lk; ws+=w[j]
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ws2=0.
        for j in range(N):
            dv=(vel[j]-ve)/max(zsig*2.,0.005)
            lz=max(np.exp(-0.5*dv*dv) if dv*dv<600. else 0.,1e-300)
            w[j]*=lz; ws2+=w[j]
        if ws2>0.:
            for j in range(N): w[j]/=ws2
        else:
            for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,vel=_resamp(pos,vel,w,N,RP,RV)
            for j in range(N): w[j]=1./N
        wm=0.
        for j in range(N): wm+=w[j]*pos[j]
        pts[i]=wm; va=0.
        for j in range(N): va+=w[j]*(pos[j]-wm)**2
        std_[i]=va**0.5; pm=md_v[i]; pz=z_v[i]
    return pts,std_


def _grid(tw_tvt,tw_gr,step=0.2):
    tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max())
    tvt_g=np.arange(tmin,tmax+step,step)
    return np.interp(tvt_g,tw_tvt,tw_gr).astype(np.float64),float(tmin),float(step)


def _gr_sig(hw,tw_tvt,tw_gr):
    kn=hw[hw['TVT_input'].notna()&hw['GR'].notna()]
    if len(kn)<20: return float(PF_GR_SIG_DEF)
    return float(np.clip(np.std(kn['GR'].values-np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)),
                          PF_GR_SIG_MIN,PF_GR_SIG_MAX))


def run_pf_ancc(hw,tw_tvt,tw_gr,N=ANCC_N):
    gs=_gr_sig(hw,tw_tvt,tw_gr)
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    ls=float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])
    tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values)
    dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
    ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr)
    # Pre-interpolate GR over the FULL well so lateral NaN gaps are filled
    # before slicing (matches training pattern; raw GR is ~32% NaN on lateral).
    gr_full = hw['GR'].interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    gr_v = gr_full.loc[ev.index].values.astype(np.float64)
    pts,std=_pf_ancc(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                      gr_v,gg,gmin,gst,
                      gs,ls,ir,N,ANCC_ALPHA,ANCC_RN,ANCC_PN,ANCC_IS,ANCC_RP,ANCC_RR,PF_RESAMP)
    return pts.astype(np.float32),std.astype(np.float32)


def run_pf_z(hw,tw_tvt,tw_gr,N=PF_N,PN=PF_PN):
    gs=_gr_sig(hw,tw_tvt,tw_gr)
    tw_s=pd.Series(tw_gr).rolling(PF_GR_WIN,center=True,min_periods=1).mean().values.astype(np.float32)
    kna=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return np.array([]),np.array([])
    dz_k=np.diff(kna['Z'].values); dvt=np.diff(kna['TVT_input'].values)
    dmd_k=np.diff(kna['MD'].values); m2=dmd_k>0
    if m2.sum()>=10:
        vz=dz_k[m2]/dmd_k[m2]; vt=dvt[m2]/dmd_k[m2]
        A=np.column_stack([vz,np.ones_like(vz)]); c,_,_,_=np.linalg.lstsq(A,vt,rcond=None)
        beta,icpt,zsig=float(c[0]),float(c[1]),max(float(np.std(vt-(c[0]*vz+c[1]))),0.001)
    else: beta,icpt,zsig=-1.,0.,0.1
    t2=kna.tail(20); dvt2=np.diff(t2['TVT_input'].values); dmd2=np.diff(t2['MD'].values); m3=dmd2>0
    iv=float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum()>=3 else 0.
    gg,gmin,gst=_grid(tw_tvt,tw_gr)
    gs2,_,_=_grid(tw_tvt,tw_s)
    gr_sm=hw['GR'].rolling(PF_GR_WIN,center=True,min_periods=1).mean()
    pts,std=_pf_z(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                   ev['GR'].values.astype(np.float64),
                   gr_sm.loc[ev.index].values.astype(np.float64),
                   gg,gs2,gmin,gst,gs,float(kna['TVT_input'].iloc[-1]),iv,
                   beta,icpt,zsig,N,
                   PF_MOM,PF_VN,PN,PF_GR_WT,PF_ROUGH_P,PF_ROUGH_V,PF_RESAMP)
    return pts.astype(np.float32),std.astype(np.float32)



# ── PF ensemble JIT (sp45 init + GR preinterp + cumulative log-lik) ───────────
@njit(cache=True)
def _pf_ancc_ll(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,ALPHA,RN,PN,IS,RP,RR,RESAMP):
    """Same as _pf_ancc but with sp45 init spread and returning total log-lik."""
    pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
    for j in range(N):
        pos[j]=ls+IS*np.random.randn()
        rate[j]=ir+0.01*np.random.randn()
    pts=np.empty(len(md_v)); pm=md_v[0]-1.; cum_ll=0.
    for i in range(len(md_v)):
        dm=md_v[i]-pm; dm=max(dm,1.)
        for j in range(N):
            rate[j]=ALPHA*rate[j]+RN*np.random.randn()
            pos[j]+=rate[j]*dm+PN*np.random.randn()
            tvt_j=pos[j]-z_v[i]
            tvt_j=max(tvt_j,vmin-50.); tvt_j=min(tvt_j,vmin+len(gg)*step+50.)
            pos[j]=tvt_j+z_v[i]
        if not np.isnan(gr_v[i]):
            ws=0.; avg_lk=0.
            for j in range(N):
                eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                d=(gr_v[i]-eg)/gs
                lk=max(np.exp(-0.5*d*d) if d*d<600. else 0.,1e-300)
                avg_lk+=w[j]*lk
                w[j]*=lk; ws+=w[j]
            cum_ll += np.log(max(avg_lk,1e-300))
            if ws>0.:
                for j in range(N): w[j]/=ws
            else:
                for j in range(N): w[j]=1./N
        ne=0.
        for j in range(N): ne+=w[j]*w[j]
        if 1./ne<RESAMP*N:
            pos,rate=_resamp(pos,rate,w,N,RP,RR)
            for j in range(N): w[j]=1./N
        tv=0.
        for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
        pts[i]=tv; pm=md_v[i]
    return pts, cum_ll


def run_pf_ensemble(hw, tw_tvt, tw_gr, n_seeds=ENS_N_SEEDS, N=ANCC_N):
    """16-seed log-lik-weighted PF ensemble, kernel pattern.

    Returns dict with one TVT array per scale + the uniform mean,
    each shaped [n_ev]. Returns None if no lateral rows.
    """
    kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev)==0 or len(kn)==0: return None
    gs = _gr_sig(hw, tw_tvt, tw_gr)
    ls = float(kn['TVT_input'].iloc[-1] + kn['Z'].iloc[-1])
    tail = kn.tail(30); dt=np.diff(tail['TVT_input'].values)
    dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    # Pre-interpolate GR over the FULL well (kernel pattern)
    gr_full = hw['GR'].interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    gr_v = gr_full.loc[ev.index].values.astype(np.float64)
    md_v = ev['MD'].values.astype(np.float64)
    z_v  = ev['Z'].values.astype(np.float64)

    preds = np.empty((n_seeds, len(ev)), dtype=np.float64)
    lls   = np.empty(n_seeds, dtype=np.float64)
    for s in range(n_seeds):
        np.random.seed(s)
        pts, ll = _pf_ancc_ll(md_v, z_v, gr_v, gg, gmin, gst,
                               gs, ls, ir, N,
                               ANCC_ALPHA, ANCC_RN, ANCC_PN, ENS_ANCC_IS, ANCC_RP, ANCC_RR, PF_RESAMP)
        # _pf_ancc_ll returns (pos - z) i.e. TVT directly when interpreted as ANCC anchor
        # (legacy contract from kernel). pts here = TVT prediction.
        preds[s] = pts
        lls[s]   = ll

    lls_n = lls - lls.max()
    out = {}
    for sc in ENS_SCALES:
        wts = np.exp(lls_n / float(sc)); wts /= wts.sum()
        out[f'pf_ens_s{int(sc)}'] = (wts[:, None] * preds).sum(0).astype(np.float32)
    out['pf_ens_mean'] = preds.mean(0).astype(np.float32)
    return out



# ── Beam search (numba JIT, 14 configs) ───────────────────────────────────────
BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2),(10,  8.0,  64.0, 2),( 8, 35.0, 220.0, 1),
    (10, 14.0,  90.0, 5),(20,  4.0,  36.0, 3),(12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2),(20, 30.0, 200.0, 2),(15, 10.0,  80.0, 4),
    (25,  6.0,  50.0, 3),(10, 40.0, 300.0, 1),(12, 18.0, 120.0, 5),
    (30,  8.0,  70.0, 2),(10, 50.0, 400.0, 0),
]


@njit(cache=True)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n=len(sgr); nt=len(tw_gr); MAX=BS*6
    bidx=np.zeros(BS,np.int64); bidx[0]=si
    bcost=np.full(BS,1e30);     bcost[0]=0.; bn=np.int64(1)
    hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
    cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
    for step in range(n):
        gv=sgr[step]; nc=np.int64(0)
        for bi in range(bn):
            idx=bidx[bi]; cost=bcost[bi]
            for d in range(-2,3):
                ni=idx+d
                if ni<0 or ni>=nt: continue
                tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)
                fnd=np.int64(-1)
                for ci in range(nc):
                    if cI[ci]==ni: fnd=ci; break
                if fnd>=0:
                    if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                else:
                    if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
        kept=min(BS,nc)
        for i in range(kept):
            mi=i
            for j in range(i+1,nc):
                if cC[j]<cC[mi]: mi=j
            if mi!=i:
                cI[i],cI[mi]=cI[mi],cI[i]
                cC[i],cC[mi]=cC[mi],cC[i]
                cP[i],cP[mi]=cP[mi],cP[i]
        hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
        bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
    best=np.int64(0)
    for b in range(1,bn):
        if bcost[b]<bcost[best]: best=b
    path=np.zeros(n,np.int64); b=best
    for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
    return path


def _nn(arr,v):
    i=int(np.searchsorted(arr,v,'left'))
    if i>=len(arr): return len(arr)-1
    if i>0 and abs(arr[i-1]-v)<=abs(arr[i]-v): return i-1
    return i


def _smooth(vals,fb,r):
    s=pd.Series(vals,dtype='float32').interpolate(limit_direction='both').fillna(fb)
    return (s.rolling(r*2+1,center=True,min_periods=1).mean() if r>0 else s).to_numpy(np.float32)


def beam_search(gr_h,tw_tvt,tw_gr,start_tvt,bs,mc,es,r):
    si=_nn(tw_tvt,start_tvt)
    sgr=_smooth(gr_h,float(np.nanmean(tw_gr)),r).astype(np.float64)
    path=_beam_jit(sgr,tw_gr.astype(np.float64),si,bs,float(mc),float(es))
    return tw_tvt[path].astype(np.float32)


def run_beam_all(hw, tw):
    kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0 or len(kn) == 0: return None, None
    last_tvt = float(kn.iloc[-1]['TVT_input'])
    tw_s = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(np.float64)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(np.float64)
    gr_all = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(np.float64)
    hgr = gr_all[ev.index]
    paths = np.zeros((len(ev), len(BEAM_CONFIGS)), dtype=np.float32)
    for k, (bs, mc, es, r) in enumerate(BEAM_CONFIGS):
        paths[:, k] = beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
    return paths, ev.index.values


# ── Base features (no formation columns) ──────────────────────────────────────
def safe_savgol(x, win=51, order=2):
    if len(x) <= win: return x.copy()
    return savgol_filter(x, win, order)


def extract_well_features(wid, data_dir):
    """Combined extractor: base + PF + Beam → joined per-lateral-row DF."""
    try:
        hw = pd.read_csv(f"{data_dir}/{wid}__horizontal_well.csv")
        tw = pd.read_csv(f"{data_dir}/{wid}__typewell.csv")
    except Exception as e:
        print(f"⚠ 井 {wid} 跳过：CSV 读取失败 ({type(e).__name__}: {e})")
        return None
    if len(hw) < 50:
        print(f"⚠ 井 {wid} 跳过：horizontal_well 行数 < 50 (={len(hw)})")
        return None
    md=hw["MD"].values; x=hw["X"].values; y=hw["Y"].values; z=hw["Z"].values
    tvt_inp = hw["TVT_input"].values
    gr_raw  = hw["GR"].values

    mask_lat = np.isnan(tvt_inp)
    if mask_lat.sum() == 0:
        print(f"⚠ 井 {wid} 跳过：无 lateral 行 (TVT_input 全部 non-NaN)")
        return None
    known = ~mask_lat
    if known.sum() < 10:
        print(f"⚠ 井 {wid} 跳过：known 段 < 10 行 (={int(known.sum())})")
        return None
    last_idx = np.flatnonzero(known)[-1]
    last_tvt = float(tvt_inp[last_idx])
    last_z = float(z[last_idx]); last_md=float(md[last_idx])
    last_x = float(x[last_idx]); last_y=float(y[last_idx])

    gr_clean = pd.Series(gr_raw).interpolate(limit_direction="both").bfill().ffill().values
    if np.all(np.isnan(gr_clean)): gr_clean = np.zeros_like(z)
    gr_smooth51 = safe_savgol(gr_clean, 51, 2)
    last_gr = float(gr_smooth51[last_idx])

    gr_s = pd.Series(gr_clean)
    rolls = {}
    for w in ROLLING_WINS:
        r = gr_s.rolling(w, center=True, min_periods=1)
        rolls[f"gr_mean_{w}"] = r.mean().values
        rolls[f"gr_std_{w}"]  = r.std().fillna(0).values

    dmd=np.gradient(md); dz=np.gradient(z); dx=np.gradient(x); dy=np.gradient(y)
    nmz = np.sqrt(dmd**2+dz**2)+1e-8; nxy=np.sqrt(dx**2+dy**2)+1e-8
    sin_dmd_dz=dz/nmz; cos_dmd_dz=dmd/nmz
    sin_dx_dy =dy/nxy; cos_dx_dy =dx/nxy

    neg_dz = np.zeros_like(z)
    for i in range(last_idx+1, len(z)):
        neg_dz[i] = neg_dz[i-1] + (-(z[i]-z[i-1]))

    lat_idx = np.flatnonzero(mask_lat)
    n_known = int(known.sum()); n_lat = int(mask_lat.sum())

    # PF (single, used as the dominant features)
    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].values.astype(np.float64)
    tw_gr  = tw_s["GR"].fillna(tw_s["GR"].mean()).values.astype(np.float64)
    np.random.seed(42)
    try:
        pf_a, pf_a_std = run_pf_ancc(hw, tw_tvt, tw_gr)
        pf_z_, pf_z_std= run_pf_z(hw,  tw_tvt, tw_gr)
    except Exception as e:
        print(f"⚠ 井 {wid} 跳过：PF 单 seed 异常 ({type(e).__name__}: {e})")
        return None
    if len(pf_a) != len(lat_idx):
        print(f"⚠ 井 {wid} 跳过：PF 单 seed 长度不匹配 (pf_a={len(pf_a)} vs lat_idx={len(lat_idx)})")
        return None

    # PF ensemble (16 seeds × 4 scales). Failure here is non-fatal — we fall
    # back to single-seed only (rare; would require a numerical blowup).
    try:
        ens = run_pf_ensemble(hw, tw_tvt, tw_gr)
    except Exception as e:
        print(f"⚠ 井 {wid} PF ensemble 异常 ({type(e).__name__}: {e})")
        ens = None
    if ens is None or len(ens['pf_ens_s12']) != len(lat_idx):
        print(f"⚠ 井 {wid} PF ensemble 回退到 single-seed")
        extract_well_features._ens_fallback_count = getattr(extract_well_features, "_ens_fallback_count", 0) + 1
        # graceful fallback: copy single-seed PF into all ensemble slots
        ens = {f'pf_ens_s{int(s)}': pf_a.astype(np.float32) for s in ENS_SCALES}
        ens['pf_ens_mean'] = pf_a.astype(np.float32)

    # Beam
    try:
        paths, _ = run_beam_all(hw, tw)
    except Exception as e:
        print(f"⚠ 井 {wid} 跳过：beam search 异常 ({type(e).__name__}: {e})")
        return None
    if paths is None or len(paths) != len(lat_idx):
        n_paths = "None" if paths is None else len(paths)
        print(f"⚠ 井 {wid} 跳过：beam search 长度不匹配 (paths={n_paths} vs lat_idx={len(lat_idx)})")
        return None

    beam_mean = paths.mean(1); beam_std = paths.std(1); beam_med=np.median(paths,1)
    beam_rng  = paths.max(1) - paths.min(1)
    beam_cons = paths[:, 0]; beam_sm5 = paths[:, 3]

    rows = []
    for k, r in enumerate(lat_idx):
        rec = {
            "well": wid, "row_idx": int(r),
            "md_offset": float(md[r]-last_md),
            "z_rel": float(z[r]-last_z),
            "x_rel": float(x[r]-last_x),
            "y_rel": float(y[r]-last_y),
            "cumsum_neg_dz": float(neg_dz[r]),
            "sin_dmd_dz": float(sin_dmd_dz[r]),
            "cos_dmd_dz": float(cos_dmd_dz[r]),
            "sin_dx_dy":  float(sin_dx_dy[r]),
            "cos_dx_dy":  float(cos_dx_dy[r]),
            "gr_smooth": float(gr_smooth51[r]),
            "gr_diff_from_last": float(gr_smooth51[r]-last_gr),
            "last_known_tvt": last_tvt,
            "last_known_z":   last_z,
            "last_known_gr":  last_gr,
            "n_known_rows":  n_known,
            "n_lateral_rows": n_lat,
            "row_position_norm": float((r-last_idx)/max(n_lat,1)),
            # PF
            "pf_ancc_std": float(pf_a_std[k]),
            "pf_z_std":    float(pf_z_std[k]),
            "pf_ancc_offset": float(pf_a[k]  - last_tvt),
            "pf_z_offset":    float(pf_z_[k] - last_tvt),
            "pf_disagreement": float(pf_a[k] - pf_z_[k]),
            "pf_mean_offset":  float(0.5*((pf_a[k]-last_tvt)+(pf_z_[k]-last_tvt))),
            # Beam
            "beam_mean_offset": float(beam_mean[k] - last_tvt),
            "beam_med_offset":  float(beam_med[k]  - last_tvt),
            "beam_cons_offset": float(beam_cons[k] - last_tvt),
            "beam_sm5_offset":  float(beam_sm5[k]  - last_tvt),
            "beam_vs_pf": float((beam_mean[k]-last_tvt) - 0.5*((pf_a[k]-last_tvt)+(pf_z_[k]-last_tvt))),
            # PF ensemble (16 seeds, 4 scales — Phase 14B)
            "pf_ens_s3_offset":   float(ens['pf_ens_s3'][k]   - last_tvt),
            "pf_ens_s5_offset":   float(ens['pf_ens_s5'][k]   - last_tvt),
            "pf_ens_s8_offset":   float(ens['pf_ens_s8'][k]   - last_tvt),
            "pf_ens_s12_offset":  float(ens['pf_ens_s12'][k]  - last_tvt),
            "pf_ens_mean_offset": float(ens['pf_ens_mean'][k] - last_tvt),
            "pf_ens_vs_ancc":      float((ens['pf_ens_s12'][k] - last_tvt) - (pf_a[k] - last_tvt)),
            "pf_ens_scale_disag":  float((ens['pf_ens_s3'][k]  - last_tvt) - (ens['pf_ens_s12'][k] - last_tvt)),
            # Keep absolutes for v9 heuristic blend (not used as model features)
            "pf_ancc_abs": float(pf_a[k]),
            "pf_ens_s12_abs": float(ens['pf_ens_s12'][k]),
        }
        for kk, arr in rolls.items(): rec[kk] = float(arr[r])
        # Target only available in train
        if not np.isnan(hw["TVT"].iloc[r] if "TVT" in hw.columns else np.nan):
            rec["target"] = float(hw["TVT"].iloc[r] - last_tvt)
        else:
            rec["target"] = np.nan
        rows.append(rec)
    return pd.DataFrame(rows)


def _warmup_numba():
    md=np.linspace(1,50,20,np.float64); z=np.zeros(20,np.float64)
    gr=np.full(20,50.,np.float64); gg=np.linspace(45,55,100,np.float64)
    _pf_ancc(md,z,gr,gg,45.,0.1,20.,50.,0.,8,0.998,0.002,0.005,0.3,0.1,0.001,0.5)
    _pf_z(md,z,gr,gr,gg,gg,45.,0.1,20.,50.,0.,-1.,0.,0.1,8,0.993,0.005,0.01,0.3,0.2,0.003,0.5)
    _pf_ancc_ll(md,z,gr,gg,45.,0.1,20.,50.,0.,8,0.998,0.002,0.005,4.5,0.1,0.001,0.5)
    _beam_jit(np.random.randn(30), np.random.randn(50), 25, 8, 15., 100.)

In [ ]:
def load_v10_offsets(test):
    diag = Path(V10_DIR) / "diagnostics"
    csv_path = diag / "test_base_predictions.csv"
    if not csv_path.exists():
        csv_path = diag / "test_base_predictions.csv.gz"
    if not csv_path.exists():
        raise FileNotFoundError(f"v10 test predictions missing under {diag}")
    v10 = pd.read_csv(csv_path)
    expected = ["id"] + V10_KEYS
    if list(v10.columns) != expected:
        raise ValueError(f"unexpected v10 columns: {v10.columns.tolist()}")
    v10["well"] = v10["id"].str.rsplit("_", n=1).str[0]
    v10["row_idx"] = v10["id"].str.rsplit("_", n=1).str[1].astype("int32")

    base = test[["well", "row_idx"]].copy()
    base["id"] = base["well"] + "_" + base["row_idx"].astype(str)
    aligned = base.merge(v10, on=["well", "row_idx", "id"], how="left", validate="one_to_one")
    missing = int(aligned[V10_KEYS].isna().any(axis=1).sum())
    print(f"  v10 alignment: rows={len(aligned):,} missing={missing}")
    if missing:
        raise ValueError(f"missing v10 predictions for {missing} rows")
    return {
        "v10_tabicl_A": aligned["tabicl_A"].to_numpy(np.float32),
        "v10_tabicl_B": aligned["tabicl_B"].to_numpy(np.float32),
    }


def load_ravaghi_offsets(test):
    import ravaghi_features

    print("  building ravaghi-schema test features")
    t0 = time.time()
    rav = ravaghi_features.build_test_features(INPUT_DIR)
    if len(rav) == 0:
        raise ValueError("ravaghi feature builder returned no rows")
    rav["well"] = rav["well"].astype(str)
    rav["row_idx"] = rav["id"].str.rsplit("_", n=1).str[1].astype("int32")

    base = test[["well", "row_idx"]].copy()
    base["_order"] = np.arange(len(base), dtype=np.int32)
    rav = base.merge(rav, on=["well", "row_idx"], how="left", validate="one_to_one").sort_values("_order")
    if len(rav) != len(test):
        raise ValueError(f"ravaghi alignment row count mismatch: {len(rav)} vs {len(test)}")
    missing_cols = [c for c in ["id", "last_known_tvt"] if c not in rav.columns or rav[c].isna().any()]
    if missing_cols:
        raise ValueError(f"ravaghi alignment missing columns/values: {missing_cols}")
    print(f"  ravaghi features: rows={len(rav):,} cols={rav.shape[1]} built in {time.time()-t0:.0f}s")

    out = {}
    for cid, rel in RAVAGHI_MODELS.items():
        pkl_path = Path(RAVAGHI_DIR) / rel
        if not pkl_path.exists():
            raise FileNotFoundError(pkl_path)
        trainer = joblib.load(pkl_path)
        feature_names = None
        for est in getattr(trainer, "estimators", []):
            if hasattr(est, "feature_names_in_"):
                feature_names = list(est.feature_names_in_)
                break
            if hasattr(est, "feature_names_"):
                feature_names = list(est.feature_names_)
                break
            if hasattr(est, "booster_"):
                feature_names = list(est.booster_.feature_name())
                break
        if not feature_names:
            feature_names = [c for c in rav.columns if c not in {"well", "id", "target", "row_idx", "_order"}]
        missing = [c for c in feature_names if c not in rav.columns]
        if missing:
            raise ValueError(f"{cid} missing {len(missing)} ravaghi features, first={missing[:10]}")
        pred = trainer.predict(rav[feature_names]).astype(np.float32)
        out[cid] = pred
        print(f"    {cid:18s} mean={pred.mean():.3f} std={pred.std():.3f} range=[{pred.min():.1f},{pred.max():.1f}]")
    return out


def main():
    t0 = time.time()
    print("=== ROGII v9: Inference-only with pre-trained models ===\n")

    # Debug: Check input directory structure
    print("[DEBUG] Checking Kaggle input directory structure:")
    if os.path.exists('/kaggle/input'):
        for item in os.listdir('/kaggle/input'):
            item_path = os.path.join('/kaggle/input', item)
            print(f"[DEBUG]   {item} -> {item_path}")
            if os.path.isdir(item_path):
                try:
                    subitems = os.listdir(item_path)
                    for subitem in subitems[:20]:
                        subitem_path = os.path.join(item_path, subitem)
                        size_str = f" ({os.path.getsize(subitem_path)} bytes)" if os.path.isfile(subitem_path) else ""
                        print(f"[DEBUG]     - {subitem}{size_str}")
                except Exception as e:
                    print(f"[DEBUG]     [ERROR listing {item_path}: {e}]")

    for label, path in [("RAVAGHI_DIR", RAVAGHI_DIR), ("V10_DIR", V10_DIR)]:
        print(f"\n[DEBUG] Checking {label}: {path}")
        try:
            for f in sorted(os.listdir(path))[:30]:
                fpath = os.path.join(path, f)
                size = os.path.getsize(fpath) if os.path.isfile(fpath) else 0
                print(f"[DEBUG]     - {f} ({size} bytes)")
        except Exception as e:
            print(f"[DEBUG]   [ERROR listing {path}: {e}]")

    print("\n[DEBUG] === End of debug info ===\n")

    print("[1/4] Numba warmup")
    np.random.seed(0); _warmup_numba()

    print(f"\n[2/4] Checking mounted artifacts")
    t1 = time.time()
    print(f"  ✓ PF feature code embedded in notebook")
    print(f"  ✓ ravaghi artifacts: {RAVAGHI_DIR}")
    print(f"  ✓ v10 artifacts: {V10_DIR}")
    print(f"  Artifact check time: {time.time()-t1:.0f}s")

    print(f"\n[3/4] Building PF128 TEST features")
    t2 = time.time()
    test_wells = sorted({f.replace("__horizontal_well.csv","")
                         for f in os.listdir(TEST_DIR)
                         if f.endswith("__horizontal_well.csv")})
    print(f"  test wells: {len(test_wells)}")
    dft = []
    for i, wid in enumerate(test_wells):
        df_w = extract_well_features(wid, TEST_DIR)
        if df_w is None:
            print(f"  ! {wid}: feature build failed"); continue
        dft.append(df_w)
        if (i+1) % 100 == 0:
            print(f"  {i+1}/{len(test_wells)} | rows so far: {sum(len(d) for d in dft):,} | "
                  f"{time.time()-t2:.0f}s", flush=True)
    test = pd.concat(dft, ignore_index=True)
    print(f"  test rows: {len(test):,} | feat build {time.time()-t2:.0f}s")
    ens_fb_test = getattr(extract_well_features, "_ens_fallback_count", 0)
    print(f"  test PF ensemble fallbacks: {ens_fb_test} wells")

    print(f"\n[4/4] Generating full run_v7 predictions")
    t3 = time.time()

    # Every component is in offset space (target = TVT - last_known_tvt), matching
    # the OOF candidates used by run_v7_with_v10_tabicl. Add last_known_tvt once.
    components = {
        "c20_r9_pf128_full": (test["pf_ens_s12_abs"].values - test["last_known_tvt"].values).astype(np.float32),
    }
    components.update(load_v10_offsets(test))
    components.update(load_ravaghi_offsets(test))

    wsum = float(sum(RUN_V7_WEIGHTS.values()))
    print(f"  raw run_v7 weight sum: {wsum:.6f}")
    blend_off = np.zeros(len(test), dtype=np.float64)
    for cid, w in RUN_V7_WEIGHTS.items():
        if cid not in components:
            raise KeyError(f"missing component: {cid}")
        arr = components[cid].astype(np.float64)
        blend_off += (w / wsum) * arr
        print(f"    {cid:20s} w={w/wsum:.4f} mean={arr.mean():.3f} std={arr.std():.3f}")

    pred_tvt = test["last_known_tvt"].values.astype(np.float64) + blend_off
    print(f"  blend offset mean: {blend_off.mean():.3f}  std: {blend_off.std():.3f}")
    print(f"  blend tvt mean: {pred_tvt.mean():.0f}  range: [{pred_tvt.min():.0f}, {pred_tvt.max():.0f}]")
    print(f"  Prediction time: {time.time()-t3:.0f}s")

    sub = pd.DataFrame({
        "id":  test["well"] + "_" + test["row_idx"].astype(str),
        "tvt": pred_tvt.astype(np.float32),
    })

    # Align to sample_submission
    sample_path = f"{INPUT_DIR}/sample_submission.csv"
    if not os.path.exists(sample_path):
        # Kaggle may mount at different depths
        for candidate in [f"/kaggle/input/rogii-wellbore-geology-prediction/sample_submission.csv",
                          f"/kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv"]:
            if os.path.exists(candidate):
                sample_path = candidate
                break
    if os.path.exists(sample_path):
        sample = pd.read_csv(sample_path)
        sub = sample[["id"]].merge(sub, on="id", how="left")
        n_missing = int(sub["tvt"].isna().sum())
        if n_missing:
            missing_ids = sub.loc[sub["tvt"].isna(), "id"].head(10).tolist()
            raise ValueError(f"missing predictions for {n_missing} sample rows, first={missing_ids}")
    sub.to_csv(OUT_PATH, index=False)
    print(f"  → {OUT_PATH}  ({len(sub)} rows)")
    print(sub.head().to_string(index=False))
    print(f"\n  pred stats: min={pred_tvt.min():.0f}, max={pred_tvt.max():.0f}, "
          f"mean={pred_tvt.mean():.0f}, median={np.median(pred_tvt):.0f}")
    print(f"\nTotal wall time: {time.time()-t0:.0f}s")


if __name__ == "__main__":
    main()